# VRAM probe -- find the largest --device-batch-size before committing hours of training

Uses `kaggle/vram_probe.py`, which builds the real GPT model + real Muon/AdamW optimizer
(same as `scripts/base_train.py`) and searches for the largest batch size that avoids a CUDA
OOM, via `accelerate.utils.find_executable_batch_size` (handles cache-clearing between OOM
retries -- a naive retry loop without that gets false negatives from memory fragmentation).

Runs in a couple of minutes per depth tested (torch.compile warmup dominates, not the actual
search) -- cheap way to de-risk a multi-hour run before starting it.

Upload via File -> Upload Notebook. T4 x2 accelerator (only GPU 0 is used -- see markdown
on Cell 2 for why one GPU is representative), internet access for `pip install accelerate`
and cloning the repo. No Kaggle Secrets needed -- this doesn't touch Google Drive.

## Cell 1: clone repo, install dependencies

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml
!uv pip install --system --python {sys.executable} accelerate

print("Cell 1 done.")

## Cell 2: probe candidate depths

Only uses GPU 0 -- DDP (what `base_train.py` actually uses across both T4s) replicates the
full model + optimizer state onto *each* GPU independently, so per-GPU memory usage depends
on the local (per-device) batch size, not on how many GPUs are in the job. Testing on one GPU
gives the same answer as testing under `torchrun --nproc_per_node=2`, for a fraction of the
setup cost.

Edit `DEPTHS_TO_TEST` to whatever you're actually considering.

In [ ]:
import os

os.chdir("/kaggle/working/repo")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

DEPTHS_TO_TEST = [5, 6, 7, 8]

for depth in DEPTHS_TO_TEST:
    print(f"\n{'='*60}\nProbing depth={depth}\n{'='*60}")
    !python kaggle/vram_probe.py --depth={depth} --max-seq-len=2048 --starting-batch-size=32